<a href="https://colab.research.google.com/github/WaitUps/CourseraExercises-AI-Agents-and-Agentic-AI-Course/blob/main/ProgrammaticPrompting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Google Generative AI SDK: Gemini Integration Demo

**Adaptation Note:** This code has been adapted and refined from an earlier version originally designed for the OpenAI model, used as part of an exercise in the following Coursera course and specialization:

*   **Course:** [AI Agents and Agentic AI with Python & Generative AI](https://www.coursera.org/learn/ai-agents-python)
*   **Specialization:** [AI Agent Developer Specialization](https://www.coursera.org/specializations/ai-agents)


This notebook serves as a practical guide and demonstration for interacting with Google's Generative AI models (like Gemini) using the Python SDK within a Colab environment. It covers essential steps and best practices for integrating powerful AI capabilities into your applications.

# Section 1: Agentic AI Exercise - Gemini Adaptation for Core API Integration and Flexible Response Generation

This section focuses on the foundational aspects of integrating with Google's Generative AI models using the Python SDK. We'll cover API key management, model discovery, and implementing a versatile helper function for generating model responses.

### 1.1 Key Concepts

**Key features and concepts demonstrated:**

*   **API Key Management:** Securely configuring your Google API Key from Colab secrets.
*   **Model Discovery:** Listing available Generative AI models.
*   **Flexible Response Generation:** Implementing a `generate_response` helper function designed to:
    *   Interact seamlessly with various Gemini models.
    *   Handle system instructions (passing them directly or integrating into chat history based on model compatibility).
    *   Format messages correctly for the Gemini API.

The `generate_response` function is a component, built for adaptability and robust handling of model interactions, including detailed examples of how to interpret model outputs.

In [26]:
from google.colab import userdata # Imports the userdata module to access secrets stored in Google Colab
import google.genai as genai # Imports the Google Generative AI SDK

# Retrieve the API key from Colab's secrets manager.
# The key should be named 'GOOGLE_API_KEY' in the secrets tab.
api_key = userdata.get('GOOGLE_API_KEY')
client = genai.Client(api_key=api_key) # Initialize the client with the API key

In [28]:
#import google.genai as genai

for m in client.models.list():
  # Check if the model supports the 'generateContent' method.
  # This is crucial for models that are intended for text generation and conversational AI.
  if "generateContent" in m.supported_actions:
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2

In [42]:
from typing import List, Dict # Used for type hinting to improve code readability and maintainability
import google.genai as genai # Imports the Google Generative AI SDK for interacting with Gemini models

# The 'client' object is expected to be globally available from the API key setup cell (KEYrzG2vB8Ip).
# It provides the direct interface for interacting with the Gemini API service.

def generate_response(messages: List[Dict]) -> str:
    """
    Calls the Gemini LLM to generate a response based on a list of messages using the direct client interface.
    It handles system instructions by incorporating them as the first 'user' message in the content list.
    """
    # Message Formatting for Gemini API: The `client.models.generate_content` method expects
    # messages in a specific format for `contents`:
    # Each message should be a dictionary like `{'role': 'user'/'model', 'parts': [{'text': '...'}]}`.
    # System instructions, if provided as 'system' role, are converted to 'user' role for this method.

    formatted_messages_for_client = []
    for msg in messages:
        # Convert 'system' role to 'user' role for direct client.models.generate_content calls,
        # as system instructions are typically passed as the first user message in this interface.
        actual_role = msg['role']
        if actual_role == 'system':
            actual_role = 'user'
        formatted_messages_for_client.append({'role': actual_role, 'parts': [{'text': msg['content']}]})

    # Content Generation using the direct client interface:
    # Instead of instantiating a GenerativeModel, we call `generate_content` directly on `client.models`.
    # The 'model' argument specifies which Gemini model to use.
    # 'contents' is the list of formatted messages forming the conversation history.
    # `generation_config` controls generation parameters like `max_output_tokens`.
    response = client.models.generate_content(
        model='gemini-flash-latest', # Specify the model directly
        contents=formatted_messages_for_client,
        config={'max_output_tokens':4096,
                #'temperature': 0.7
                }
    )
    # Return Value:
    # The generated text content from the model's response is extracted and returned.
    return response.text

In [43]:
messages = [
    {"role": "system", "content": "You are an expert software engineer that prefers functional programming."},
    {"role": "user", "content": "Write a function to swap the keys and values in a dictionary."}
]

response = generate_response(messages)
print(response)

In functional programming, we treat data as immutable and focus on transformations. To swap keys and values in a dictionary, we want to transform a collection of pairs $(k, v)$ into a collection of pairs $(v, k)$.

Here are a few ways to achieve this in Python, ranging from the idiomatic "Pythonic" functional style to a more rigorous approach that handles potential data loss.

### 1. The Declarative Approach (Dictionary Comprehension)
This is the most common way to perform a swap. It is declarative because it describes the relationship between the old and new data structures.

```python
from typing import Dict, TypeVar

K = TypeVar('K')
V = TypeVar('V')

def swap_dict(d: Dict[K, V]) -> Dict[V, K]:
    """Returns a new dictionary with keys and values swapped."""
    return {v: k for k, v in d.items()}
```

### 2. The Higher-Order Function Approach
If we want to avoid explicit loops entirely, we can use `map` and `reversed`. Since `d.items()` returns a sequence of `(key, value)` tuples, 

## Sample of Generated Response (Agentic AI Exercise: Section 1):

*Note: This displays the response generated by the AI model as a Markdown preview. Each execution of the `generate_response` function may produce a different output.*

<br>

---
<br>

In functional programming, we treat data as immutable and focus on transformations. To swap keys and values in a dictionary, we want to transform a collection of pairs $(k, v)$ into a collection of pairs $(v, k)$.

Here are a few ways to achieve this in Python, ranging from the idiomatic "Pythonic" functional style to a more rigorous approach that handles potential data loss.

### 1. The Declarative Approach (Dictionary Comprehension)
This is the most common way to perform a swap. It is declarative because it describes the relationship between the old and new data structures.

```python
from typing import Dict, TypeVar

K = TypeVar('K')
V = TypeVar('V')

def swap_dict(d: Dict[K, V]) -> Dict[V, K]:
    """Returns a new dictionary with keys and values swapped."""
    return {v: k for k, v in d.items()}
```

### 2. The Higher-Order Function Approach
If we want to avoid explicit loops entirely, we can use `map` and `reversed`. Since `d.items()` returns a sequence of `(key, value)` tuples, we can reverse each tuple to get `(value, key)`.

```python
def swap_dict_functional(d: Dict[K, V]) -> Dict[V, K]:
    # reversed() on a tuple (k, v) returns an iterator yielding v then k
    # dict() can consume an iterable of iterables (pairs)
    return dict(map(reversed, d.items())) # type: ignore
```

### 3. Handling Collisions (The "Fold" Approach)
In a standard swap, if two keys have the same value, one will be overwritten (last-one-in wins). A true functional engineer considers edge cases: **What if the mapping is not injective?**

To avoid losing data, we can use a "fold" (known as `reduce` in Python) to group keys into a list for each value.

```python
from functools import reduce
from typing import List

def swap_and_group(d: Dict[K, V]) -> Dict[V, List[K]]:
    """
    Swaps keys and values, grouping keys into lists to prevent data loss.
    This is a classic 'fold' pattern.
    """
    def accumulator(acc: Dict[V, List[K]], item: tuple[K, V]) -> Dict[V, List[K]]:
        key, value = item
        # In a purely functional language, we'd return a new dict here
        # To be performant in Python, we update the accumulator
        acc.setdefault(value, []).append(key)
        return acc

    return reduce(accumulator, d.items(), {})

# Example:
# input: {'a': 1, 'b': 2, 'c': 1}
# output: {1: ['a', 'c'], 2: ['b']}
```

### Which one should you use?
1.  **Use Example 1** for 99% of use cases. It is readable, efficient, and idiomatic.
2.  **Use Example 2** if you are strictly adhering to a "point-free" or piping style common in languages like Haskell or F#.
3.  **Use Example 3** if your dictionary values are not unique and you cannot afford to lose data during the transformation.

**Note on Purity:** In all these examples, the original dictionary `d` remains unchanged, satisfying the core FP principle of immutability.



# Section 2: Building a Quasi-Agent for Python Function Generation

This section implements a 'quasi-agent' that demonstrates how to maintain context across multiple prompts with an LLM to build a Python function step-by-step. It asks the user for a function description, then uses sequential calls to the `generate_response` function (our LLM wrapper) to:

1.  Generate the basic Python function.
2.  Add comprehensive documentation to the function.
3.  Generate `unittest` test cases for the function.

Key improvements:
- The `develop_custom_function` leverages a growing message history to iteratively refine the generated Python function, documentation, and test cases.
- This allows for multi-turn interactions where subsequent LLM calls build upon previous responses, improving the coherence and completeness of the generated artifacts.
- Facilitates a more robust and 'agentic' development workflow by extending the agent's memory.

This exercise highlights managing information flow and handling LLM outputs, even when they might be inconsistent. We will leverage the `generate_response` function we defined earlier, which is configured to use `gemini-flash-latest` and handles system instructions and message formatting.

In [44]:
from typing import List, Dict
import sys
import os

def extract_code_block(response: str) -> str:
   """Extracts the first Python code block from a string response."""

   if '```' not in response:
      # If no code block markers, assume the entire response is code or not what we expect
      # For robustness, we might want to return an empty string or raise an error here
      return response.strip()

   # Split by ``` and take the content between the first two markers
   parts = response.split('```')
   if len(parts) < 3:
      # Not enough parts to contain a code block
      return response.strip()

   code_block = parts[1].strip()

   # Remove 'python' or 'py' if it's at the start of the code block
   if code_block.startswith("python"): # Covers 'python' and 'python\n'
      code_block = code_block[6:]
   elif code_block.startswith("py"): # Covers 'py' and 'py\n'
      code_block = code_block[2:]

   return code_block.strip()

In [45]:
def develop_custom_function():
   import io # Import the io module
   import sys # Import the sys module

   # Get user input for function description BEFORE redirecting stdout
   print("\nWhat kind of Python function would you like to create?")
   print("Example: 'A function that calculates the factorial of a number'")
   print("Your description: ", end='')
   function_description = input().strip()

   # Redirect stdout to a StringIO object to capture all subsequent print statements
   old_stdout = sys.stdout
   redirected_output = io.StringIO()
   sys.stdout = redirected_output

   try:
     # Initialize conversation with a system prompt. Our generate_response handles system messages.
     messages = [
        {"role": "system", "content": "You are a Python expert helping to develop a function. Always output code within a ```python code block```."}
     ]

     print("\n--- STEP 1: Generating Initial Function ---")
     # First prompt - Basic function
     user_prompt_1 = f"Write a basic Python function that {function_description}."
     messages.append({"role": "user", "content": user_prompt_1})

     llm_raw_response_1 = generate_response(messages)
     initial_function_code = extract_code_block(llm_raw_response_1)

     print("\n=== Initial Function (LLM Response) ===")
     print(initial_function_code)

     # Add LLM's response to conversation history.
     # We feed back the extracted code wrapped in markdown, as seen in the exercise example,
     # to provide clear context of what was 'generated' in the previous step.
     messages.append({"role": "model", "content": f"```python\n{initial_function_code}\n```"})

     print("\n--- STEP 2: Adding Documentation ---")
     # Second prompt - Add documentation
     user_prompt_2 = "Now, add comprehensive documentation to this function, including a description, parameter descriptions, return value description, example usage, and edge cases. Make sure the output is just the updated function in a ```python code block```."
     messages.append({"role": "user", "content": user_prompt_2})

     llm_raw_response_2 = generate_response(messages)
     documented_function_code = extract_code_block(llm_raw_response_2)

     print("\n=== Documented Function (LLM Response) ===")
     print(documented_function_code)

     # Add documented function response to conversation history
     messages.append({"role": "model", "content": f"```python\n{documented_function_code}\n```"})

     print("\n--- STEP 3: Adding Test Cases ---")
     # Third prompt - Add test cases
     user_prompt_3 = "Finally, add unittest test cases for this function. Tests should cover basic functionality, edge cases, and various input scenarios. Output the test cases code in a separate ```python code block```."
     messages.append({"role": "user", "content": user_prompt_3})

     llm_raw_response_3 = generate_response(messages)
     test_cases_code = extract_code_block(llm_raw_response_3)

     print("\n=== Test Cases (LLM Response) ===")
     print(test_cases_code)

     # Generate a filename from the function description
     filename_base = function_description.lower().replace('a function that ', '').replace('a function to ', '').strip()
     filename_base = ''.join(c for c in filename_base if c.isalnum() or c.isspace())
     filename_base = filename_base.replace(' ', '_')
     filename = f"{filename_base[:30] if len(filename_base) > 30 else filename_base}.py"

     # Ensure the filename is not empty or just an extension
     if not filename_base:
         filename = "generated_function.py"

     # Save final version to a Python file
     full_code = documented_function_code + '\n\n' + test_cases_code
     with open(filename, 'w') as f:
        f.write(full_code)

     print(f"\nFinal code has been saved to: {filename}")
     print("\n--- Quasi-Agent Development Process Complete ---")

     return documented_function_code, test_cases_code, filename, redirected_output.getvalue()
   finally:
     # Restore stdout
     sys.stdout = old_stdout

### Run the Quasi-Agent

Execute the cell below to start the quasi-agent. You will be prompted to enter a description for the Python function you want to create. The agent will then proceed through the steps of generating the function, adding documentation, and creating test cases, printing its progress along the way.

In [46]:
# Call the quasi-agent to develop a custom function
function_code, tests, filename, function_output = develop_custom_function()


What kind of Python function would you like to create?
Example: 'A function that calculates the factorial of a number'
Your description: Converts a decimal number into its binary, octal, and hexadecimal.


In [47]:
print(f"\nFinal code has been saved to {filename}")
print(function_output)


Final code has been saved to converts_a_decimal_number_into.py

--- STEP 1: Generating Initial Function ---

=== Initial Function (LLM Response) ===
def convert_decimal(decimal_num):
    """
    Converts a decimal number into binary, octal, and hexadecimal.
    
    Args:
        decimal_num (int): The decimal integer to convert.
        
    Returns:
        dict: A dictionary containing the conversions as strings.
    """
    conversions = {
        "binary": bin(decimal_num),      # Prefix '0b'
        "octal": oct(decimal_num),       # Prefix '0o'
        "hexadecimal": hex(decimal_num)  # Prefix '0x'
    }
    return conversions

# Example usage:
number = 255
results = convert_decimal(number)

print(f"Decimal: {number}")
print(f"Binary: {results['binary']}")
print(f"Octal: {results['octal']}")
print(f"Hexadecimal: {results['hexadecimal']}")

--- STEP 2: Adding Documentation ---

=== Documented Function (LLM Response) ===
def convert_decimal(decimal_num):
    """
    Converts a de